# 05 · Feature research, retraining evidence and decisions

**This notebook uses the current notebook 02 and 03 run records.** It does not read the historical Kaggle-score CSV as current performance. The old 120-fold compact experiment remains a historical comparator, not the result of rebuilding notebook 02.

The questions are specific: Which basketball signals improve development Brier? Did the revised model search use the exact rebuilt matrix? Do ranking features help boosted trees? What should be tested next? All displayed development seasons have been explored before; none is relabeled an untouched holdout.

In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Markdown, display
from march_mania.notebook_support import table, style
from march_mania.publication.workflow import evidence, lineage

ROOT = Path.cwd() if (Path.cwd() / "pyproject.toml").exists() else Path.cwd().parent
FEATURES, FEATURE_RECORD = evidence(ROOT, "feature_store")
MODELS, MODEL_RECORD = evidence(ROOT, "model_comparison")
style()
links = lineage(ROOT)
table(links.assign(Run=links.Run.str[:16]))
display(Markdown(f"**Current feature matrix:** `{MODEL_RECORD['manifest']['inputs']['data']['features.parquet']}`  \n"
                 f"**Model fits:** {MODEL_RECORD['summary']['fit_tasks']:,}; "
                 f"**physical validation games:** {MODEL_RECORD['summary']['physical_validation_games']:,}."))

## Feature quality, not feature count

The 20 contextual additions test plausible mechanisms: ball control and defensive activity, opponent/venue and schedule context, and scoring balance/downside risk. They extend the 104-feature store to 124 columns. The fixed-estimator ablation below isolates feature changes; the nested model comparison later also tunes feature block and regularization.

Only earlier-season development evidence can justify a change. A weak full-feature model can indicate noisy, redundant or unstable signals, not insufficient dimensionality. Permutation importance describes model reliance rather than a causal basketball effect.

In [ ]:
feature_scores = pd.read_csv(FEATURES / "leaderboard.csv")
display(Markdown("**Feature ablation scoreboard:** lower mean-season Brier is better."))
table(feature_scores.sort_values(["Gender", "macro_season_brier"]).groupby("Gender", sort=False).head(8))

## Did the retraining actually change predictions?

The comparison pairs old and revised forecasts by tournament, season, physical game, route and model. Unchanged reference streams can legitimately reproduce exactly even when additional candidates are trained. New model families must be assessed separately. A historical Kaggle score cannot change merely because a feature notebook was rerendered.

In [ ]:
comparison_path = MODELS / "comparison.csv"
if comparison_path.exists():
    comparison = pd.read_csv(comparison_path)
    table(comparison[["Gender", "block", "model", "games", "old_brier", "new_brier", "max_probability_change"]])
    display(Markdown(f"**Largest reference-stream probability change:** {comparison.max_probability_change.max():.3g}. "
                     "The paired table tests reproducibility; it is not evidence that all newly added models are identical."))
else:
    display(Markdown("A paired previous-versus-current retraining record has not yet been published. Historical scores are not substituted."))

## Ranking information in nonlinear models

The earlier 497-fit search gave men's external-ranking features only to ranking logistic regression. Its common boosting comparison deliberately excluded all 23 ranking-derived columns. The revised search adds **56 actual fits**: XGBoost and LightGBM, compact ranking versus full feature blocks, two depths, seven forward seasons. The outer scoreboard covers five development seasons; all selection/calibration uses earlier seasons.

The common-feature separate/pooled blend remains unchanged so the added ranking information does not silently contaminate that comparison.

In [ ]:
scores = pd.read_csv(MODELS / "leaderboard.csv")
ranked = scores.loc[(scores.Gender == "M") & (scores.block == "M") &
                    scores.model.isin(["rank_logistic_raw", "rank_xgboost_raw", "rank_lightgbm_raw",
                                       "xgboost_raw", "lightgbm_raw", "logistic_raw"])].sort_values("macro_season_brier")
table(ranked[["model", "macro_season_brier", "brier", "log_loss", "roc_auc", "games"]])
fig, ax = plt.subplots(figsize=(8.5, 4.4), constrained_layout=True)
ax.barh(ranked.model.str.replace("_raw", "").str.replace("_", " "), ranked.macro_season_brier)
ax.invert_yaxis()
ax.set(title="Men · ranking-augmented models on matched development seasons",
       xlabel="Mean season Brier · lower is better", xlim=(0, max(.25, ranked.macro_season_brier.max()*1.1)))
for i, value in enumerate(ranked.macro_season_brier):
    ax.text(value + .002, i, f"{value:.4f}", va="center", fontsize=10)
ax.spines[["top", "right"]].set_visible(False)
plt.show()

In [ ]:
seasonal = pd.read_csv(MODELS / "metrics_by_season.csv")
chosen = seasonal.loc[(seasonal.Gender == "M") & (seasonal.block == "M") &
                      seasonal.model.isin(["rank_logistic_raw", "rank_xgboost_raw", "rank_lightgbm_raw"])]
fig, ax = plt.subplots(figsize=(8.5, 4.4), constrained_layout=True)
for model, group in chosen.groupby("model"):
    ax.plot(group.Season, group.brier, marker="o", label=model.replace("_raw", "").replace("_", " "))
ax.set(title="Does the ranking effect persist across seasons?", xlabel="Outer validation season",
       ylabel="Brier · lower is better", xticks=sorted(chosen.Season.unique()))
ax.spines[["top", "right"]].set_visible(False)
ax.legend(frameon=False)
plt.show()
reference = ranked.loc[ranked.model == "rank_logistic_raw", "macro_season_brier"].iloc[0]
new = ranked.loc[ranked.model.isin(["rank_xgboost_raw", "rank_lightgbm_raw"])]
if len(new):
    best = new.iloc[0]
    change = float(best.macro_season_brier - reference)
    display(Markdown(f"**Measured change versus ranking logistic:** {best.model} = {change:+.6f} mean-season Brier. "
                     + ("The added ranking-tree hypothesis did not beat the ranking-logistic reference." if change >= 0 else
                        "This development minimum is promising, but not an untouched-test improvement or automatic final promotion.")))
else:
    display(Markdown("The ranking-tree extension is implemented but has not yet been fitted in this evidence snapshot."))

## Decision and the next experiment

Retain the complete candidate evidence, including disappointing models. Do not call the largest feature block the best model. Diagnose unstable families using the 02 ablations and the 03 held-out permutation and season diagnostics. The next bounded test should compare opponent-adjusted style matchups and a margin-regression challenger on the same temporal development contract, with prior-only selection and calibration. Missing injury/rotation data stays explicitly unavailable; it must not be fabricated.

Historical PyTorch/TensorFlow and margin-regression scores were not rerun here. Those are separate remaining experiments, not completed capabilities. No result in this notebook proves global state-of-the-art performance.

**To generate your own file:** notebook 04 contains the opt-in refit/inference and download cell. The code never uploads to Kaggle. The [old score log](../reports/submission_portfolio/kaggle_scores.csv) remains historical evidence, not the headline for this run.